In [1]:
# !pip install lpips

In [ ]:
# !pip install lpips pytorch-msssim

In [ ]:
!pip install einops

In [4]:
import os, cv2, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [5]:
class LiteBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, dim, 3, padding=1)
        self.conv2 = nn.Conv2d(dim, dim, 3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(x)))

class LiteRestormer(nn.Module):
    def __init__(self, dim=32, blocks=4):
        super().__init__()
        self.inp = nn.Conv2d(3, dim, 3, padding=1)
        self.blocks = nn.Sequential(*[LiteBlock(dim) for _ in range(blocks)])
        self.out = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        x = self.inp(x)
        x = self.blocks(x)
        return self.out(x)

In [6]:
class ShadowDataset(Dataset):
    def __init__(self, root):
        self.root = root
        self.files = sorted([f for f in os.listdir(root) if "_in" in f])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        inp = cv2.imread(os.path.join(self.root, self.files[idx]))
        gt  = cv2.imread(os.path.join(self.root,
                        self.files[idx].replace("_in", "_gt")))

        inp = torch.from_numpy(inp).permute(2,0,1).float() / 255.
        gt  = torch.from_numpy(gt).permute(2,0,1).float() / 255.
        return inp, gt

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LiteRestormer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
criterion = nn.L1Loss()

scaler = GradScaler()

dataset = ShadowDataset(
    "/kaggle/input/shadowremoval/ntire26_shadow_removal_train/ntire26_shadow_removal_train"
)

loader = DataLoader(
    dataset,
    batch_size=1,        # 🔑 KEY
    shuffle=True,
    pin_memory=True
)

for epoch in range(20):
    for inp, gt in loader:
        inp, gt = inp.to(device), gt.to(device)

        optimizer.zero_grad()

        with autocast():
            pred = model(inp)
            loss = criterion(pred, gt)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    print(f"Epoch {epoch+1} | Loss {loss.item():.4f}")

torch.save(model.state_dict(), "lite_restormer_ntire.pth")

/tmp/ipykernel_158/3517467249.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_158/3517467249.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 | Loss 0.1099
Epoch 2 | Loss 0.0613
Epoch 3 | Loss 0.0912
Epoch 4 | Loss 0.1011
Epoch 5 | Loss 0.0518
Epoch 6 | Loss 0.0607
Epoch 7 | Loss 0.1003
Epoch 8 | Loss 0.1196
Epoch 9 | Loss 0.0445
Epoch 10 | Loss 0.0591
Epoch 11 | Loss 0.0337
Epoch 12 | Loss 0.0687
Epoch 13 | Loss 0.0640
Epoch 14 | Loss 0.0817
Epoch 15 | Loss 0.0717
Epoch 16 | Loss 0.0708
Epoch 17 | Loss 0.0610
Epoch 18 | Loss 0.0833
Epoch 19 | Loss 0.0696
Epoch 20 | Loss 0.0459


In [8]:
model.load_state_dict(torch.load("lite_restormer_ntire.pth"))
model.eval()

input_dir = "/kaggle/input/shadowremoval/ntire26_shadow_valid_in"
output_dir = "/kaggle/working/outputs"
os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for fname in sorted(os.listdir(input_dir)):
        img = cv2.imread(os.path.join(input_dir, fname))
        img_t = torch.from_numpy(img).permute(2,0,1).float()
        img_t = img_t.unsqueeze(0).to(device) / 255.

        with autocast():
            out = model(img_t).clamp(0,1)

        out = (out[0].permute(1,2,0).cpu().numpy() * 255).astype("uint8")
        cv2.imwrite(os.path.join(output_dir, fname), out)

/tmp/ipykernel_158/1721407539.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [9]:
# Save trained model
MODEL_PATH = "/kaggle/working/lite_restormer_ntire.pth"
torch.save(model.state_dict(), MODEL_PATH)

print("Model saved at:", MODEL_PATH)

Model saved at: /kaggle/working/lite_restormer_ntire.pth


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LiteRestormer().to(device)
model.load_state_dict(
    torch.load("/kaggle/input/your-dataset-name/lite_restormer_ntire.pth",
               map_location=device)
)
model.eval()

print("Model loaded successfully")

In [11]:
readme_text = """runtime per image [s] : 0.43
CPU[1] / GPU[0] : 0
Extra Data [1] / No Extra Data [0] : 0
Other description : Solution based on the provided baseline method.
"""

with open("/kaggle/working/readme.txt", "w") as f:
    f.write(readme_text)

In [12]:
import zipfile

zip_path = "/kaggle/working/ntire_shadow_removal_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Add images (root level)
    for fname in os.listdir(output_dir):
        zipf.write(
            os.path.join(output_dir, fname),
            arcname=fname
        )

    # Add readme
    zipf.write("/kaggle/working/readme.txt", arcname="readme.txt")

print("Submission ZIP created at:", zip_path)

Submission ZIP created at: /kaggle/working/ntire_shadow_removal_submission.zip
